# Skin Lesion Triage — Colab Training Notebook

MSU Final Year Project — Aleck Mudyanadzo (R252845M) & Primrose S. Ncube (R253036M)

**Before you start:**
1. Runtime -> Change runtime type -> GPU (T4 is fine).
2. Have your `skin_lesion_triage.zip` project folder ready to upload.
3. Have your Kaggle API token (`kaggle.json`) ready — get it from kaggle.com/settings -> API -> Create New Token.

This notebook: uploads your project -> downloads HAM10000 -> preps the data -> trains MobileNetV2 and ResNet50 -> lets you download the trained `.keras` files to drop into your local `models/` folder.

## 1. Upload your project zip

In [ ]:
from google.colab import files
print('Select your skin_lesion_triage.zip file...')
uploaded = files.upload()

In [ ]:
import zipfile, glob, os

zip_name = [f for f in uploaded.keys() if f.endswith('.zip')][0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

%cd /content/skin_lesion_triage
!ls

## 2. Install dependencies

In [ ]:
!pip install -q kaggle opencv-python-headless
# TensorFlow, numpy, pandas, scikit-learn already come preinstalled on Colab

## 3. Upload your Kaggle API token

In [ ]:
print('Select your kaggle.json file...')
kaggle_upload = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle token installed.')

## 4. Download and prepare HAM10000

In [ ]:
!python scripts/prepare_data.py --kaggle_download

## 5. Train MobileNetV2 (fast, deploy-friendly)
Target: validation recall >= 0.85 on the malignant class (Objective 2).

In [ ]:
%cd scripts
!python train_model.py --arch mobilenetv2 --epochs_head 10 --epochs_finetune 15
%cd ..

## 6. Train ResNet50 (for comparison, per Objective 2)

In [ ]:
%cd scripts
!python train_model.py --arch resnet50 --epochs_head 10 --epochs_finetune 15
%cd ..

## 7. Download the trained models
Drop both `.keras` files into your local project's `models/` folder.

In [ ]:
from google.colab import files
import os

for fname in os.listdir('models'):
    if fname.endswith('.keras'):
        print('Downloading', fname)
        files.download(os.path.join('models', fname))

## 8. (Optional) Save models straight to Google Drive instead
Useful if the direct download above times out for large files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dest = '/content/drive/MyDrive/skin_lesion_triage_models'
os.makedirs(dest, exist_ok=True)
for fname in os.listdir('models'):
    if fname.endswith('.keras'):
        shutil.copy(os.path.join('models', fname), dest)
        print('Copied', fname, 'to Drive:', dest)